In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
 
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 130
 
# ---------------------------------------------------------------
# 1. LOAD DATA
# ---------------------------------------------------------------
FACT_PATH = r'C:\JOB_market_trend_analysis\cleaned_data\job_postings_cleaned_v4.csv'
COMPANY_PATH = r'C:\JOB_market_trend_analysis\cleaned_data\company_dim_cleaned.csv'
SKILLS_PATH = r'C:\JOB_market_trend_analysis\cleaned_data\skills_dim_cleaned.csv'
OUT_DIR = r'C:\JOB_market_trend_analysis\cleaned_data\skills_job_cleaned.csv'

df = pd.read_csv(FACT_PATH)
comp = pd.read_csv(COMPANY_PATH)
skills = pd.read_csv(SKILLS_PATH)
 
df['job_posted_date'] = pd.to_datetime(
    df['job_posted_date'], format='%d-%m-%Y %H:%M', errors='coerce'
)
 
# Subset with disclosed salary only — used for ALL salary charts
sal = df[df['salary_standardized'].notna()].copy()
print(f"Total postings: {len(df):,}")
print(f"Postings with disclosed salary: {len(sal):,} "
      f"({len(sal) / len(df) * 100:.1f}%)")
 
# ---------------------------------------------------------------
# 2. EXECUTIVE OVERVIEW (2x2 grid)
# ---------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f"Executive Overview — Job Market Analysis (n={len(df):,} postings)",
             fontsize=14, fontweight='bold')
 
sns.histplot(sal['salary_standardized'], bins=50, kde=True,
             ax=axes[0, 0], color="#4C72B0")
axes[0, 0].set_title(
    f"Salary Distribution (n={len(sal):,} disclosed, "
    f"{len(sal) / len(df) * 100:.1f}% of postings)"
)
axes[0, 0].set_xlabel("Standardized Yearly Salary ($)")
 
wfh_counts = df['work_mode'].value_counts()
axes[0, 1].pie(wfh_counts, labels=wfh_counts.index, autopct='%1.1f%%',
                colors=["#DD8452", "#4C72B0"], startangle=90)
axes[0, 1].set_title("Remote vs Hybrid/In-Office Split")
 
trend = df.groupby(df['job_posted_date'].dt.to_period('M')).size()
trend.index = trend.index.astype(str)
axes[1, 0].plot(trend.index, trend.values, marker='o', color="#55A868")
axes[1, 0].set_title("Job Postings Trend by Month")
axes[1, 0].tick_params(axis='x', rotation=75)
axes[1, 0].set_ylabel("Postings Count")
 
sen_counts = df['seniority_level'].value_counts()
sns.barplot(x=sen_counts.values, y=sen_counts.index,
            hue=sen_counts.index, palette="Blues_r", legend=False, ax=axes[1, 1])
axes[1, 1].set_title("Postings by Seniority Level")
axes[1, 1].set_xlabel("Count")
 
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(OUT_DIR + '1_executive_overview.png', bbox_inches='tight')
plt.close()
 
# ---------------------------------------------------------------
# 3. TOP PAYING JOB TITLES
# ---------------------------------------------------------------
top_paying = (
    sal.groupby('job_title_short')['salary_standardized']
    .agg(['mean', 'count'])
    .query('count >= 20')                      # avoid noisy small samples
    .sort_values('mean', ascending=False)
    .head(10)
)
 
plt.figure(figsize=(10, 6))
sns.barplot(x=top_paying['mean'], y=top_paying.index,
            hue=top_paying.index, palette="mako", legend=False)
plt.title("Top 10 Highest-Paying Roles (avg standardized salary, min 20 postings)")
plt.xlabel("Average Salary ($)")
plt.ylabel("")
for i, (v, c) in enumerate(zip(top_paying['mean'], top_paying['count'])):
    plt.text(v + 1000, i, f"${v:,.0f}  (n={c})", va='center', fontsize=9)
plt.tight_layout()
plt.savefig(OUT_DIR + '2_top_paying_jobs.png', bbox_inches='tight')
plt.close()
 
# ---------------------------------------------------------------
# 4. TOP HIRING COMPANIES
# ---------------------------------------------------------------
merged = df.merge(comp[['company_id', 'name']], on='company_id', how='left')
top_companies = merged['name'].value_counts().head(10)
 
plt.figure(figsize=(10, 6))
sns.barplot(x=top_companies.values, y=top_companies.index,
            hue=top_companies.index, palette="crest", legend=False)
plt.title("Top 10 Companies by Number of Job Postings")
plt.xlabel("Number of Postings")
plt.tight_layout()
plt.savefig(OUT_DIR + '3_top_hiring_companies.png', bbox_inches='tight')
plt.close()
 
# ---------------------------------------------------------------
# 5. SALARY BY SENIORITY & WORK MODE
# ---------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
 
order = (
    sal.groupby('seniority_level')['salary_standardized']
    .median().sort_values(ascending=False).index
)
sns.boxplot(data=sal, x='salary_standardized', y='seniority_level',
            order=order, hue='seniority_level', palette="Blues",
            legend=False, ax=axes[0])
axes[0].set_title("Salary Distribution by Seniority Level")
axes[0].set_xlabel("Salary ($)")
 
sns.boxplot(data=sal, x='salary_standardized', y='work_mode',
            hue='work_mode', palette="Oranges", legend=False, ax=axes[1])
axes[1].set_title("Salary Distribution: Remote vs Hybrid/In-Office")
axes[1].set_xlabel("Salary ($)")
 
plt.tight_layout()
plt.savefig(OUT_DIR + '4_salary_by_seniority_workmode.png', bbox_inches='tight')
plt.close()
 
# ---------------------------------------------------------------
# 6. HIRING DEMAND TIER
# ---------------------------------------------------------------
plt.figure(figsize=(8, 5))
tier_order = (
    ['Low', 'Medium', 'High']
    if set(['Low', 'Medium', 'High']).issubset(set(df['hiring_demand_tier'].unique()))
    else df['hiring_demand_tier'].value_counts().index
)
sns.countplot(data=df, x='hiring_demand_tier', order=tier_order,
              hue='hiring_demand_tier', palette="viridis", legend=False)
plt.title("Postings by Monthly Hiring-Demand Tier")
plt.xlabel("")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(OUT_DIR + '5_hiring_demand_tier.png', bbox_inches='tight')
plt.close()
 
# ---------------------------------------------------------------
# 7. SKILLS CATALOG OVERVIEW (skills_dim only — catalog, not usage)
# ---------------------------------------------------------------
plt.figure(figsize=(9, 5))
type_counts = skills['type'].value_counts()
sns.barplot(x=type_counts.values, y=type_counts.index,
            hue=type_counts.index, palette="rocket", legend=False)
plt.title("Skill Catalog: Count of Distinct Skills by Type\n"
          "(catalog only — not usage frequency; bridge table needed for that)")
plt.xlabel("Number of Distinct Skills")
plt.tight_layout()
plt.savefig(OUT_DIR + '6_skills_catalog_by_type.png', bbox_inches='tight')
plt.close()
 
print("All 6 charts saved to", OUT_DIR)
 
# =================================================================
# PLACEHOLDER — SKILLS DEMAND & SKILLS-VS-SALARY
# =================================================================
# Uncomment and adapt once you have the job<->skill bridge table
# (needs job_id, skill_id columns — e.g. skills_job_dim.csv).
#
# bridge = pd.read_csv('/mnt/user-data/uploads/skills_job_dim.csv')
#
# # ---- Most in-demand skills (frequency across ALL postings) ----
skill_freq = (
    bridge.merge(skills, on='skill_id')
    ['skills'].value_counts().head(10)
 )
# plt.figure(figsize=(10, 6))
# sns.barplot(x=skill_freq.values, y=skill_freq.index,
#             hue=skill_freq.index, palette="flare", legend=False)
# plt.title("Top 10 Most In-Demand Skills (by posting frequency)")
# plt.xlabel("Number of Postings Mentioning Skill")
# plt.tight_layout()
# plt.savefig(OUT_DIR + '7_top_in_demand_skills.png', bbox_inches='tight')
# plt.close()
#
# # ---- Highest paying skills (join bridge -> fact for salary) ----
# skill_salary = (
#     bridge.merge(sal[['job_id', 'salary_standardized']], on='job_id')
#     .merge(skills, on='skill_id')
#     .groupby('skills')['salary_standardized']
#     .agg(['mean', 'count'])
#     .query('count >= 20')
#     .sort_values('mean', ascending=False)
#     .head(10)
# )
# plt.figure(figsize=(10, 6))
# sns.barplot(x=skill_salary['mean'], y=skill_salary.index,
#             hue=skill_salary.index, palette="mako", legend=False)
# plt.title("Top 10 Highest-Paying Skills (avg salary, min 20 postings)")
# plt.xlabel("Average Salary ($)")
# plt.tight_layout()
# plt.savefig(OUT_DIR + '8_top_paying_skills.png', bbox_inches='tight')
# plt.close()
#
# # ---- Optimal skills: demand vs salary quadrant ----
# demand_salary = (
#     bridge.merge(sal[['job_id', 'salary_standardized']], on='job_id', how='left')
#     .merge(skills, on='skill_id')
#     .groupby('skills')
#     .agg(demand=('job_id', 'count'),
#          avg_salary=('salary_standardized', 'mean'))
#     .dropna()
# )
# plt.figure(figsize=(10, 8))
# sns.scatterplot(data=demand_salary, x='demand', y='avg_salary')
# plt.axvline(demand_salary['demand'].median(), linestyle='--', color='grey')
# plt.axhline(demand_salary['avg_salary'].median(), linestyle='--', color='grey')
# for skill, row in demand_salary.iterrows():
#     plt.annotate(skill, (row['demand'], row['avg_salary']), fontsize=7)
# plt.title("Optimal Skills: Demand vs. Average Salary")
# plt.xlabel("Demand (number of postings)")
# plt.ylabel("Average Salary ($)")
# plt.tight_layout()
# plt.savefig(OUT_DIR + '9_optimal_skills_quadrant.png', bbox_inches='tight')
# plt.close()

Total postings: 787,686
Postings with disclosed salary: 32,699 (4.2%)
All 6 charts saved to C:\JOB_market_trend_analysis\cleaned_data\skills_job_cleaned.csv


NameError: name 'bridge' is not defined

In [4]:
# ===============================================================
# JOB MARKET TREND ANALYSIS - EDA DASHBOARD
# ===============================================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os


# ---------------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------------

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 130


# ---------------------------------------------------------------
# PATHS
# ---------------------------------------------------------------

FACT_PATH = r'C:\JOB_market_trend_analysis\cleaned_data\job_postings_cleaned_v4.csv'

COMPANY_PATH = r'C:\JOB_market_trend_analysis\cleaned_data\company_dim_cleaned.csv'

SKILLS_PATH = r'C:\JOB_market_trend_analysis\cleaned_data\skills_dim_cleaned.csv'

# Optional bridge table
SKILL_JOB_PATH = r'C:\JOB_market_trend_analysis\cleaned_data\skills_job_dim_cleaned.csv'


# Output folder for charts
OUT_DIR = r'C:\JOB_market_trend_analysis\charts\\'

os.makedirs(OUT_DIR, exist_ok=True)



# ---------------------------------------------------------------
# 1. LOAD DATA
# ---------------------------------------------------------------

df = pd.read_csv(FACT_PATH)

comp = pd.read_csv(COMPANY_PATH)

skills = pd.read_csv(SKILLS_PATH)



print("Job postings shape:", df.shape)
print("Company shape:", comp.shape)
print("Skills shape:", skills.shape)



# ---------------------------------------------------------------
# CHECK COLUMNS
# ---------------------------------------------------------------

print("\nJob Columns:")
print(df.columns.tolist())


# ---------------------------------------------------------------
# DATE CLEANING
# ---------------------------------------------------------------

df['job_posted_date'] = pd.to_datetime(
    df['job_posted_date'],
    format='%d-%m-%Y %H:%M',
    errors='coerce'
)



# ---------------------------------------------------------------
# SALARY DATA
# ---------------------------------------------------------------

if 'salary_standardized' not in df.columns:
    raise Exception("salary_standardized column not found")


sal = df[
    df['salary_standardized'].notna()
].copy()



print("\nTotal postings:", f"{len(df):,}")

print(
    "Postings with disclosed salary:",
    f"{len(sal):,}",
    f"({len(sal)/len(df)*100:.1f}%)"
)



# ===============================================================
# 2. EXECUTIVE OVERVIEW
# ===============================================================


fig, axes = plt.subplots(
    2,
    2,
    figsize=(14,10)
)


fig.suptitle(
    f"Executive Overview - Job Market Analysis (n={len(df):,})",
    fontsize=14,
    fontweight='bold'
)



# Salary Distribution

sns.histplot(
    sal['salary_standardized'],
    bins=50,
    kde=True,
    ax=axes[0,0]
)


axes[0,0].set_title(
    f"Salary Distribution\n"
    f"Disclosed Salaries Only ({len(sal):,})"
)

axes[0,0].set_xlabel(
    "Standardized Yearly Salary ($)"
)



# Work Mode

wfh_counts = df['work_mode'].value_counts()


axes[0,1].pie(
    wfh_counts.values,
    labels=wfh_counts.index,
    autopct='%1.1f%%',
    startangle=90
)


axes[0,1].set_title(
    "Remote vs Hybrid/In-Office"
)



# Hiring Trend

trend = (
    df.groupby(
        df['job_posted_date'].dt.to_period('M')
    )
    .size()
)


trend.index = trend.index.astype(str)



axes[1,0].plot(
    trend.index,
    trend.values,
    marker='o'
)


axes[1,0].tick_params(
    axis='x',
    rotation=75
)


axes[1,0].set_title(
    "Monthly Job Posting Trend"
)


axes[1,0].set_ylabel(
    "Number of Jobs"
)



# Seniority

sen_counts = df['seniority_level'].value_counts()


sns.barplot(
    x=sen_counts.values,
    y=sen_counts.index,
    ax=axes[1,1]
)


axes[1,1].set_title(
    "Jobs by Seniority Level"
)



plt.tight_layout()


plt.savefig(
    OUT_DIR + "1_executive_overview.png",
    bbox_inches='tight'
)

plt.close()



# ===============================================================
# 3. TOP PAYING JOB TITLES
# ===============================================================


top_paying = (

    sal.groupby('job_title_short')
    ['salary_standardized']
    .agg(['mean','count'])

    .query("count >= 20")

    .sort_values(
        'mean',
        ascending=False
    )

    .head(10)

)



plt.figure(figsize=(10,6))


sns.barplot(
    data=top_paying,
    x='mean',
    y=top_paying.index
)


plt.title(
    "Top 10 Highest Paying Job Roles"
)


plt.xlabel(
    "Average Salary ($)"
)


plt.ylabel("")



plt.tight_layout()


plt.savefig(
    OUT_DIR+"2_top_paying_jobs.png",
    bbox_inches='tight'
)


plt.close()



# ===============================================================
# 4. TOP HIRING COMPANIES
# ===============================================================


merged = df.merge(
    comp[['company_id','name']],
    on='company_id',
    how='left'
)



top_companies = (
    merged['name']
    .value_counts()
    .head(10)
)



plt.figure(figsize=(10,6))


sns.barplot(
    x=top_companies.values,
    y=top_companies.index
)


plt.title(
    "Top Hiring Companies"
)


plt.xlabel(
    "Number of Job Postings"
)



plt.tight_layout()


plt.savefig(
    OUT_DIR+"3_top_hiring_companies.png",
    bbox_inches='tight'
)


plt.close()



# ===============================================================
# 5. SALARY BY SENIORITY & WORK MODE
# ===============================================================


fig,axes = plt.subplots(
    1,
    2,
    figsize=(14,6)
)



sns.boxplot(
    data=sal,
    x='salary_standardized',
    y='seniority_level',
    ax=axes[0]
)


axes[0].set_title(
    "Salary by Seniority"
)



sns.boxplot(
    data=sal,
    x='salary_standardized',
    y='work_mode',
    ax=axes[1]
)


axes[1].set_title(
    "Salary: Remote vs Office"
)



plt.tight_layout()


plt.savefig(
    OUT_DIR+"4_salary_analysis.png",
    bbox_inches='tight'
)


plt.close()



# ===============================================================
# 6. HIRING DEMAND
# ===============================================================


plt.figure(figsize=(8,5))


sns.countplot(
    data=df,
    x='hiring_demand_tier'
)


plt.title(
    "Hiring Demand Tier"
)



plt.tight_layout()


plt.savefig(
    OUT_DIR+"5_hiring_demand.png",
    bbox_inches='tight'
)


plt.close()



# ===============================================================
# 7. SKILLS CATALOG
# ===============================================================


plt.figure(figsize=(9,5))


type_counts = skills['type'].value_counts()


sns.barplot(
    x=type_counts.values,
    y=type_counts.index
)


plt.title(
    "Skill Catalog Distribution"
)


plt.xlabel(
    "Number of Skills"
)



plt.tight_layout()


plt.savefig(
    OUT_DIR+"6_skill_catalog.png",
    bbox_inches='tight'
)


plt.close()



# ===============================================================
# 8. OPTIONAL SKILL DEMAND ANALYSIS
# ===============================================================


if os.path.exists(SKILL_JOB_PATH):


    bridge = pd.read_csv(SKILL_JOB_PATH)


    skill_freq = (

        bridge
        .merge(
            skills,
            on='skill_id'
        )

        ['skills']
        .value_counts()
        .head(10)

    )


    plt.figure(figsize=(10,6))


    sns.barplot(
        x=skill_freq.values,
        y=skill_freq.index
    )


    plt.title(
        "Top 10 Most In-Demand Skills"
    )


    plt.xlabel(
        "Number of Job Postings"
    )


    plt.tight_layout()


    plt.savefig(
        OUT_DIR+"7_top_skills.png",
        bbox_inches='tight'
    )


    plt.close()


    print("Skill demand analysis completed")

else:

    print(
        "skills_job_dim file not found - skipped skill demand analysis"
    )



# ===============================================================
# FINAL
# ===============================================================


print("\n====================================")
print("ALL ANALYSIS COMPLETED SUCCESSFULLY")
print("Charts saved at:")
print(OUT_DIR)
print("====================================")

Job postings shape: (787686, 29)
Company shape: (140030, 6)
Skills shape: (252, 4)

Job Columns:
['job_id', 'company_id', 'job_title_short', 'job_title', 'job_location', 'job_via', 'job_schedule_type', 'job_work_from_home', 'search_location', 'job_posted_date', 'job_no_degree_mention', 'job_health_insurance', 'job_country', 'salary_rate', 'salary_year_avg', 'salary_hour_avg', 'salary_standardized', 'salary_type', 'salary_outlier', 'seniority_level', 'work_mode', 'posted_year', 'posted_month', 'posted_quarter', 'skill_count_per_posting', 'postings_count_by_month', 'month_hiring_rank', 'is_high_demand_month', 'hiring_demand_tier']

Total postings: 787,686
Postings with disclosed salary: 32,699 (4.2%)
skills_job_dim file not found - skipped skill demand analysis

ALL ANALYSIS COMPLETED SUCCESSFULLY
Charts saved at:
C:\JOB_market_trend_analysis\charts\\
